In [ ]:
%%time 

# autoload external python modules if they changed
%load_ext autoreload
%autoreload 2

# show plots inline in a notebook (no extra spaces/characters are allowed after 'inline')
%matplotlib inline

# add ../funcs to the current path
import sys, os
sys.path.append(os.path.join(os.getcwd(), "../funcs")) 

# Change Matplotlib's Default DPI
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 100  # 100 for histogram, 250 for contours

from base import query_dataset, query_data
from jdiag import load_jdiag, get_jdiag_metadata, get_jdiag_data, get_valid_subset
from fit_rate import fit_rate
from histogram import histogram
import numpy as np

import bokeh
print(bokeh.__version__)
from bokeh.plotting import output_notebook, figure, show
from bokeh.models import HoverTool
# Enable inline plotting in Jupyter
output_notebook()

In [ ]:
%%time

# read data from netcdf files, save them to 'dataset'
#dataset = load_jdiag("../data/samples/mpasjedi/jdiag_aircar_t133.nc")
dataset = load_jdiag("../data/samples/mpasjedi/conus12km/jdiag_aircar_t133.nc")
#dataset = load_jdiag("/lfs6/BMC/wrfruc/gge/OPSROOT/cyc_12km/backup2/stmp/20240527/rrfs_jedivar_05_v2.0.9/det/jedivar_05/jdiag_t133.nc4")
# dataset = load_jdiag("/gpfs/f6/bil-fire10-oar/world-shared/gge/OPSROOT/cyc_12km/stmp/20240527/rrfs_jedivar_01_v2.1.0/det/jedivar_01/jdiag_aircar_t133.nc")
#dataset = load_jdiag("/gpfs/f6/bil-fire10-oar/world-shared/gge/OPSROOT/cyc_12km/stmp/20240527/rrfs_getkf_observer_00_v2.1.0/enkf/jdiag_aircar_q133.nc")
#dataset = load_jdiag("/gpfs/f6/bil-fire10-oar/world-shared/gge/OPSROOT/cyc_12km/stmp/20240527/rrfs_jedivar_01_v2.0.9/det/jedivar_01.reject/jdiag_aircar_t133.nc")
#dataset1 = load_jdiag("/gpfs/f6/bil-fire10-oar/world-shared/gge/OPSROOT/cyc_12km/com/rrfs/v2.0.9/rrfs.20240527/00/ioda_bufr/det/ioda_adpsfc.nc")
#dataset2 = load_jdiag("/gpfs/f6/bil-fire10-oar/world-shared/gge/OPSROOT/cyc_12km/com/rrfs/v2.0.9/rrfs.20240527/00/ioda_bufr/det/ioda_sfcshp.nc")
query_dataset(dataset)

# dataset = load_jdiag("/gpfs/f6/bil-fire10-oar/world-shared/gge/OPSROOT/standalone/t183_DA/oneobserver/jdiag_adpsfc_t183.nc")
# dataset1 = load_jdiag("/gpfs/f6/bil-fire10-oar/world-shared/gge/OPSROOT/standalone/t183_DA/twoobservers/jdiag_adpsfc_t183.nc")
# dataset2 = load_jdiag("/gpfs/f6/bil-fire10-oar/world-shared/gge/OPSROOT/standalone/t183_DA/twoobservers/jdiag_sfcshp_t183.nc")

#metadata = get_jdiag_metadata(dataset)
#query_data(metadata)
#data = get_jdiag_data(dataset, "specificHumidity")
data = get_jdiag_data(dataset, "airTemperature")
#data1 = get_jdiag_data(dataset1, "airTemperature")
#data2 = get_jdiag_data(dataset2, "airTemperature")
# data1 = get_jdiag_data(dataset1, "airTemperature")
# data2 = get_jdiag_data(dataset2, "airTemperature")
# vdata = get_valid_subset(data, "oman", condition={"EffectiveQC2":0} )
# vdata1 = get_valid_subset(data1, "oman", condition={"EffectiveQC2":0} )
# vdata2 = get_valid_subset(data2, "oman", condition={"EffectiveQC2":0} )
# print(len(vdata))
# print(len(vdata1))
# print(len(vdata2))

# data = get_jdiag_data(dataset, "specificHumidity")
# data = get_jdiag_data(dataset, "airTemperature")

#query_data(data)
#np.set_printoptions(threshold=np.inf)
#print(len(data['DerivedObsError']))
# print(len(data['ObsValue']))
# print(len(data1['ObsValue']))
# print(len(data2['ObsValue']))

In [ ]:
np.set_printoptions(threshold=np.inf)
query_data(data1)
subdata1 = get_valid_subset(data1, "dumpReportType", condition={"ObsType":183})
subdata2 = get_valid_subset(data2, "dumpReportType", condition={"ObsType":183})
query_data(subdata1)
query_data(subdata2)
#query_dataset(dataset)
#print(data["EffectiveQC2"])

In [ ]:
# print out values in the `data` dictionary
query_data(data)
#values = data['EffectiveError2']*1000
#print(', '.join(f"{x:.2f}" for x in values))

In [ ]:
%%time
histogram(data['oman'], bin_size=0.1, n_xticks=25) #, xlabel='airTemperature', title='t133 histogram')

In [ ]:
# query_data(data)
fit_rate(data, dz=1000)

In [ ]:
%%time
bin_size = 0.2
data_array = data['oman']
data_array = data_array[~np.isnan(data_array)]
hist, edges = np.histogram(data['ombg'], bins=np.arange(np.nanmin(data['ombg']), np.nanmax(data['ombg']) + bin_size, bin_size))
p = figure(title="T133 Bokeh Air Temperature Histogram", x_axis_label='Air Temperature (C)', 
           y_axis_label='Frequency (Hz)', tools="pan,box_zoom,reset,save")
hover = HoverTool(tooltips=[("Frequency (Hz)", "@top"), ("Range", "@left to @right")])
p.add_tools(hover)
p.quad(top=hist, bottom=0, left=edges[:-1], right=edges[1:], fill_color="navy", line_color="white", alpha=0.7)
# Show interactive plot
show(p)